# HandScriptEval AI — Qwen2.5-VL-3B GPU OCR Server (Google Colab)
Run all cells on a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).
Copy the printed `https://....trycloudflare.com` URL and set it on your local server!

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils
!pip install -q --upgrade-strategy only-if-needed pymupdf qwen-vl-utils transformers accelerate fastapi uvicorn python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    major, _ = torch.cuda.get_device_capability()
    dtype = torch.bfloat16 if major >= 8 else torch.float16
else:
    dtype = torch.float32

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
min_pixels = 512 * 28 * 28
max_pixels = 1280 * 28 * 28

if "vlm_model" not in globals():
    vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        vlm_model = vlm_model.to("cpu")
    vlm_model.eval()

    vlm_proc = AutoProcessor.from_pretrained(
        model_id,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
    )
    vlm_proc.tokenizer.padding_side = "left"

print(f"Loaded {model_id} on {device.upper()} ({dtype})")

In [ ]:
import io, re, time, threading, subprocess
import torch, uvicorn
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from qwen_vl_utils import process_vision_info

app = FastAPI()

PROMPT = (
    "You are a precise OCR engine. Transcribe every word, number, and symbol "
    "from this page—both printed and handwritten—in natural reading order. "
    "Preserve exact question numbers, headings, and line breaks. "
    "Do not summarize, translate, or guess illegible words; mark unclear words as [?]. "
    "Output only the transcribed text."
)

@app.get("/health")
def health():
    return {"status": "ok", "device": device}

@app.post("/ocr")
async def ocr_endpoint(file: UploadFile = File(...)):
    t0 = time.time()
    raw = await file.read()
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
    texts = [vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)]
    image_inputs, _ = process_vision_info(messages)
    inputs = vlm_proc(text=texts, images=image_inputs, padding=True, return_tensors="pt").to(device)
    with torch.inference_mode():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=1280, do_sample=False, use_cache=True)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    text = vlm_proc.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    return {"text": text, "seconds": round(time.time() - t0, 2)}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning"), daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "="*60)
        print("COPY THIS URL TO YOUR LOCAL SERVER:")
        print(m.group(0))
        print("="*60 + "\n")
        break
